## Defining the dataframe 
- Due to the edition of Databricks, getting .csv data through spark is not doable. 
- Therefore, defining dataframe through pandas and convert it to spark dataframe through **_spark.createDataFrame()_**

In [0]:
# df = spark.read.option("header",True).option("inferSchema",True).csv("/Workspace/Users/hhshin9879@gmail.com/energy-consumption-pipeline/demo/global_energy_consumption.csv")
import pandas as pd

df = pd.read_csv("/Workspace/Users/hhshin9879@gmail.com/energy-consumption-pipeline/data/world_energy_consumption.csv")
spark_df = spark.createDataFrame(df)

In [0]:
display(spark_df)

### Showing columns

In [0]:
spark_df.columns

## Select the columns which are going to be used
- Country-level greenhouse gas emissions prediction
based on energy consumption and macroeconomic features
- Lag features (`emissions_lag1`, `emissions_lag2`) were added to incorporate historical emission values, enabling the model to capture temporal trends.

In [0]:
from pyspark.sql.functions import col

col_list = [
    "country",
    "year",
    "population",
    "gdp",
    "coal_consumption",
    "oil_consumption",
    "gas_consumption",
    "renewables_consumption",
    "greenhouse_gas_emissions"
]

df_clean = spark_df.select(*col_list)
display(df_clean)

In [0]:
countries = [
    "United States",
    "China",
    "India",
    "Sweden"
]

# The data ends at 2019
df_countries = (
    df_clean
    .filter(col("country").isin(countries))
    .filter((col("year") >= 2000) & (col("year") <= 2019))
    .dropna()
)

display(df_countries)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window = Window.partitionBy("country").orderBy("year")

df_countries = df_countries.withColumn("emissions_lag1", lag("greenhouse_gas_emissions", 1).over(window))
df_countries = df_countries.withColumn("emissions_lag2", lag("greenhouse_gas_emissions", 2).over(window))

df_countries = df_countries.dropna()
display(df_countries)